In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import DeltaTable
import sys

## Utilities 

In [0]:
class SilverTransformation:
    def __init__(self, df=None):
        self.df = df

    def read_bronzedata(self, dataset):
        self.df = spark.read.table(f"walmart_catalog.bronze.{dataset}")
        return self.df

    def add_currtimestamp(self, cdc_col):
        self.df = self.df.withColumn(cdc_col, current_timestamp())
        return self.df

    def fill_nulls(self):
        self.df = self.df.fillna(0)
        return self.df
    
    def fill_strnulls(self):
        self.df = self.df.fillna("N/A")
        return self.df
    
    def drop_specificcols(self, col_val):
        self.df = self.df.drop(*col_val)
        return self.df
    
    def split_values (self, col_val, existent_col, delimeter, index):
        self.df = self.df.withColumn(col_val, split(col(existent_col), delimeter)[index])
        return self.df
    

    def scd_type1(self, table_val, primary_key, cdc_column ):
        from delta.tables import DeltaTable

        if spark.catalog.tableExists(f"walmart_catalog.silver.{table_val}"):
            dlt_obj = DeltaTable.forName(spark, f"walmart_catalog.silver.{table_val}")
            dlt_obj.alias("t").merge(
                self.df.alias("s"),
                f"t.{primary_key} = s.{primary_key}",
            ).whenMatchedUpdateAll(condition=f"s.{cdc_column} > t.{cdc_column}")\
                .whenNotMatchedInsertAll()\
                .execute()
        else:
            self.df.write.format("delta").mode("append").option("path", f"abfss://silver@walmartlakejess.dfs.core.windows.net/{table_val}_Data/{table_val}").saveAsTable(f"walmart_catalog.silver.{table_val}")
        return self.df
    
    def drop_duplicatedata(self, primary_key):
        self.df = self.df.dropDuplicates([primary_key])
        return self.df
    
    def cast_intcols(self, int_cols):
        for cols in int_cols:
            self.df = self.df.withColumn(cols, col(cols).cast(IntegerType()))
        return self.df
    
    def cast_timestampcols(self, timestamp_cols):
        for cols in timestamp_cols:
            self.df = self.df.withColumn(cols, col(cols).cast(TimestampType()))
        return self.df

    def drop_specific_cols(self, data_values):
        self.df = self.df.drop(*data_values)
        return self.df
    
    def cast_doublecols(self, double_cols):
        for cols in double_cols:
            self.df = self.df.withColumn(cols, col(cols).cast(DoubleType()))
        return self.df
    
    def rename_columns(self, column_mapping: dict):
        for old_name, new_name in column_mapping.items():
            self.df = self.df.withColumnRenamed(old_name, new_name)
        return self.df
    

In [0]:
sys.path.append(
    "/Workspace/Users/jessepepple36@gmail.com/walmart_project/utils"
)



## Product Enrichment

In [0]:
products_obj = SilverTransformation()

In [0]:
df_products = products_obj.read_bronzedata("products")
df_products = products_obj.drop_specific_cols(["_rescued_data", "is_active", "updated_timestamp", "created_timestamp"])
df_products = products_obj.cast_intcols(["product_id"])
df_products = products_obj.fill_strnulls()
df_products = products_obj.split_values("product_flag", "product_name", " ", 0)
df_products = products_obj.drop_duplicatedata("product_id")
df_products = products_obj.add_currtimestamp("last_updated_timestamp")
df_products = products_obj.cast_doublecols(["price"])
df_products = products_obj.scd_type1("products", "product_id", "last_updated_timestamp")

df_products.display()

## Customer Enrichment

In [0]:
customers_obj = SilverTransformation()

In [0]:
df_customers = customers_obj.read_bronzedata("customers")
df_customers = customers_obj.rename_columns({"first_name": "customer_firstname","last_name": "customer_lastname","email": "customer_email","phone": "customer_phone","city": "customer_city","province": "customer_province","country": "customer_country"})
df_customers = customers_obj.cast_intcols(["customer_id"])
df_customers = customers_obj.drop_duplicatedata("customer_id")
df_customers = customers_obj.drop_specificcols(["created_timestamp", "updated_timestamp", "is_active", "_rescued_data"])
df_customers = customers_obj.add_currtimestamp("last_updated_timestamp")
df_customers = customers_obj.fill_strnulls()
df_customers = customers_obj.fill_nulls()
df_customers = customers_obj.scd_type1("customers", "customer_id", "last_updated_timestamp")

df_customers.display()

## Emloyees Enrichment

In [0]:
employee_obj = SilverTransformation()

In [0]:
df_employees = employee_obj.read_bronzedata("employees")
df_employees.display()

In [0]:
df_employees = employee_obj.read_bronzedata("employees")
df_employees = employee_obj.cast_intcols(["employee_id", "store_id"])
df_employees = employee_obj.cast_doublecols(["salary"])
df_employees = employee_obj.rename_columns({"first_name" : "employee_firstname", "last_name" : "employee_lastname", "email" : "employee_email", "job_title" : "employee_jobtitle", "salary" : "employee_salary"})
df_employees = employee_obj.drop_specificcols(["_rescued_data", "created_timestamp", "updated_timestamp", "is_active"])
df_employees = employee_obj.drop_duplicatedata("employee_id")
df_employees = employee_obj.add_currtimestamp("last_updated_timestamp")
df_employees = employee_obj.fill_strnulls()
df_employees = employee_obj.fill_nulls()
df_employees = employee_obj.scd_type1("employees", "employee_id", "last_updated_timestamp")
df_employees.display()

## Stores Enrichment

In [0]:
stores_obj = SilverTransformation()

In [0]:
df_stores = stores_obj.read_bronzedata("stores")
df_stores = stores_obj.cast_intcols(["store_id"])
df_stores = stores_obj.rename_columns({"city" : "store_city", "province" : "store_province", "country" : "store_country"})
df_stores = stores_obj.drop_specific_cols(["_rescued_data", "created_timestamp", "updated_timestamp", "is_active"])
df_stores = stores_obj.drop_duplicatedata("store_id")
df_stores = stores_obj.add_currtimestamp("last_updated_timestamp")
df_stores = stores_obj.fill_nulls()
df_stores = stores_obj.fill_strnulls()
df_stores = df_stores.withColumn("store_number",regexp_extract(col("store_name"), r"(Store \d+)", 1))
stores_obj.df = df_stores
df_stores = stores_obj.scd_type1("stores", "store_id", "last_updated_timestamp")

df_stores.display()

## ORDERS Enrichment

In [0]:
orders_obj = SilverTransformation()

In [0]:
df_orders = orders_obj.read_bronzedata("orders")
df_orders = orders_obj.cast_intcols(["order_id", "customer_id", "store_id"])
df_orders = orders_obj.cast_timestampcols(["order_timestamp"])
df_orders = orders_obj.cast_doublecols(["total_amount"])
df_orders = orders_obj.add_currtimestamp("last_updated_timestamp")
df_orders = orders_obj.fill_strnulls()
df_orders = orders_obj.fill_nulls()
df_orders = orders_obj.drop_specific_cols(["_rescued_data", "created_timestamp", "updated_timestamp", "is_active"])
df_orders = orders_obj.drop_duplicatedata("order_id")
df_orders = orders_obj.scd_type1("orders", "order_id", "last_updated_timestamp")
df_orders.display()

## ORDER ITEMS ENRICHMENT

In [0]:
orderitems_obj = SilverTransformation()



In [0]:
df_orderitems = orderitems_obj.read_bronzedata("order_items")
df_orderitems = orderitems_obj.cast_intcols(["order_item_id", "order_id", "product_id", "quantity"])
df_orderitems = orderitems_obj.cast_doublecols(["unit_price", "line_amount"])
df_orderitems = orderitems_obj.add_currtimestamp("last_updated_timestamp")
df_orderitems = orderitems_obj.drop_specific_cols(["_rescued_data", "created_timestamp", "updated_timestamp", "is_active"])
df_orderitems = orderitems_obj.drop_duplicatedata("order_item_id")
df_orderitems = orderitems_obj.fill_nulls()
df_orderitems = orderitems_obj.fill_strnulls()
df_orderitems = orderitems_obj.scd_type1("order_items", "order_item_id", "last_updated_timestamp")

df_orderitems.display()